# 02. Дерево решений вручную: 10 объектов, 2 признака

| | |
|---|---|
| **Уровень** | 1 из 4 — крошечные данные: всё считается вручную (`1_tiny`) |
| **Скрипт-источник** | [`02_tree_by_hand.py`](../02_tree_by_hand.py) — тот же код, запуск из терминала |
| **Время выполнения** | ≈ 1 с |

**Цель.** Понять, как дерево выбирает признак и порог, — главный «строительный блок» бустинга.

### Чему вы научитесь

- Перебор всех пар (признак, порог)
- Критерий «уменьшение суммы квадратов»
- Рекурсия до глубины 2
- Значения листьев — средние
- Сверка с `gbcourse.RegressionTree`

### Данные

10 квартир, придуманных для примера: площадь (м²), этаж, цена (млн). Цена растёт с площадью, а первый этаж дешевле — дерево должно найти и то и другое.

### План

1. **Этап 1.** Данные
2. **Этап 2.** Лучшее разбиение корня: перебираем оба признака и все пороги
3. **Этап 3.** Разбиения детей (глубина 2)
4. **Этап 4.** Дерево и его прогнозы
5. **Этап 4б.** Остатки первого дерева и второе дерево — идея бустинга
6. **Этап 5.** Сверка с gbcourse.RegressionTree(max_depth=2)

> Ноутбук собран из `examples/1_tiny/02_tree_by_hand.py` командой `python examples/build_notebooks.py`. Чтобы изменить пример, правьте скрипт и пересоберите ноутбук.

## Подготовка

Ячейка ниже находит папку `examples/` (ноутбук можно открыть из любой рабочей папки внутри
репозитория), подключает общий каркас примеров `_common.Example` и учебную библиотеку курса
`gbcourse`. Каркас печатает этапы и выводы, «раскрывает» датасет и показывает графики прямо в
ноутбуке. Выполняйте ячейки сверху вниз: **Run → Run All Cells**.

In [1]:
import sys
from pathlib import Path

EXAMPLES = next(p / "examples" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "examples" / "_common.py").is_file())
sys.path.insert(0, str(EXAMPLES))
from _common import Example

ex = Example(EXAMPLES / "1_tiny/02_tree_by_hand.py", title="02. Дерево решений вручную: 10 объектов, 2 признака",
             goal="увидеть, как дерево выбирает признак и порог")

import numpy as np
from gbcourse import RegressionTree

## Этап 1. Данные

In [2]:
area = np.array([30, 35, 42, 48, 55, 60, 68, 75, 82, 95], dtype=float)
floor = np.array([1, 3, 1, 5, 2, 1, 7, 4, 1, 9], dtype=float)
price = np.array([4.1, 5.0, 3.6, 6.6, 7.2, 5.4, 9.0, 9.6, 7.7, 12.4])
X = np.c_[area, floor]
ex.describe(X, price, names=["площадь", "этаж"], target="цена, млн", source="придуман для примера")
names = ["площадь", "этаж"]


def best_split(rows):
    """Перебор всех признаков и порогов: наибольшее уменьшение суммы квадратов отклонений (SSE)."""
    y = price[rows]
    sse_parent = np.sum((y - y.mean()) ** 2)
    best = None
    for j in range(2):
        xs = np.unique(X[rows, j])
        for t in (xs[:-1] + xs[1:]) / 2:
            left = rows[X[rows, j] <= t]
            right = rows[X[rows, j] > t]
            sse = np.sum((price[left] - price[left].mean()) ** 2) + np.sum((price[right] - price[right].mean()) ** 2)
            gain = sse_parent - sse
            if best is None or gain > best[0]:
                best = (gain, j, t, left, right)
    return best

   Источник: придуман для примера
   Объектов: 10   признаков: 2   память: 0.00 МБ
   Типы признаков: float64 × 2
   Пропусков нет
   Цель «цена, млн»: среднее 7.060, σ 2.590, мин 3.600, макс 12.400
   Первые строки:


,площадь,этаж,"[цена, млн]"
0,30.0,1.0,4.1
1,35.0,3.0,5.0
2,42.0,1.0,3.6


## Этап 2. Лучшее разбиение корня: перебираем оба признака и все пороги

In [3]:
root = np.arange(10)
print(f"   SSE корня (все 10 объектов вокруг среднего {price.mean():.2f}): {np.sum((price - price.mean()) ** 2):.3f}")
for j in range(2):
    xs = np.unique(X[:, j])
    gains = []
    for t in (xs[:-1] + xs[1:]) / 2:
        l_, r_ = price[X[:, j] <= t], price[X[:, j] > t]
        gains.append((t, np.sum((price - price.mean()) ** 2) - np.sum((l_ - l_.mean()) ** 2) - np.sum((r_ - r_.mean()) ** 2)))
    top = max(gains, key=lambda g: g[1])
    print(f"   {names[j]:8s}: {len(gains)} порогов, лучший {top[0]:5.1f} уменьшает SSE на {top[1]:.3f}")
gain, j, t, left, right = best_split(root)
print(f"   ⇒ корень: {names[j]} ≤ {t}  (уменьшение SSE {gain:.3f})")

   SSE корня (все 10 объектов вокруг среднего 7.06): 67.104
   площадь : 9 порогов, лучший  64.0 уменьшает SSE на 45.588
   этаж    : 6 порогов, лучший   3.5 уменьшает SSE на 36.504
   ⇒ корень: площадь ≤ 64.0  (уменьшение SSE 45.588)


## Этап 3. Разбиения детей (глубина 2)

In [4]:
tree = {"root": (j, t)}
for side, rows in (("слева", left), ("справа", right)):
    g2, j2, t2, l2, r2 = best_split(rows)
    tree[side] = (j2, t2, price[l2].mean(), price[r2].mean())
    print(f"   {side} (n = {len(rows)}) → {names[j2]} ≤ {t2} (уменьшение SSE {g2:.3f}); "
          f"листья {price[l2].mean():.3f} (n = {len(l2)}) и {price[r2].mean():.3f} (n = {len(r2)})")
ex.note("""Этаж тоже информативен (в корне он уменьшил бы SSE на 36.5), но площадь сильнее — и в корне, и в детях.
Дереву глубины 2 не хватило «места», чтобы учесть этаж. Посмотрим, что останется ему в остатках.""")

   слева (n = 6) → площадь ≤ 45.0 (уменьшение SSE 7.042); листья 4.233 (n = 3) и 6.400 (n = 3)
   справа (n = 4) → площадь ≤ 88.5 (уменьшение SSE 9.901); листья 8.767 (n = 3) и 12.400 (n = 1)


> ▸ **Вывод.** Этаж тоже информативен (в корне он уменьшил бы SSE на 36.5), но площадь сильнее — и в корне, и в детях. Дереву глубины 2 не хватило «места», чтобы учесть этаж. Посмотрим, что останется ему в остатках.

## Этап 4. Дерево и его прогнозы

In [5]:
def predict(row):
    j0, t0 = tree["root"]
    j2, t2, a, b = tree["слева" if row[j0] <= t0 else "справа"]
    return a if row[j2] <= t2 else b


pred = np.array([predict(r) for r in X])
for r, p, yv in zip(X, pred, price):
    print(f"   площадь {r[0]:4.0f}, этаж {r[1]:1.0f}: прогноз {p:6.3f}, факт {yv:5.2f}")
print(f"   MSE дерева на обучении: {np.mean((price - pred) ** 2):.4f} (у константы — {np.var(price):.4f})")

   площадь   30, этаж 1: прогноз  4.233, факт  4.10
   площадь   35, этаж 3: прогноз  4.233, факт  5.00
   площадь   42, этаж 1: прогноз  4.233, факт  3.60
   площадь   48, этаж 5: прогноз  6.400, факт  6.60
   площадь   55, этаж 2: прогноз  6.400, факт  7.20
   площадь   60, этаж 1: прогноз  6.400, факт  5.40
   площадь   68, этаж 7: прогноз  8.767, факт  9.00
   площадь   75, этаж 4: прогноз  8.767, факт  9.60
   площадь   82, этаж 1: прогноз  8.767, факт  7.70
   площадь   95, этаж 9: прогноз 12.400, факт 12.40
   MSE дерева на обучении: 0.4573 (у константы — 6.7104)


## Этап 4б. Остатки первого дерева и второе дерево — идея бустинга

In [6]:
resid = price - pred
print("   остатки: " + "  ".join(f"{v:+.2f}" for v in resid))
best2 = None
for j in range(2):
    xs = np.unique(X[:, j])
    for t in (xs[:-1] + xs[1:]) / 2:
        lft, rgt = resid[X[:, j] <= t], resid[X[:, j] > t]
        sse = np.sum((lft - lft.mean()) ** 2) + np.sum((rgt - rgt.mean()) ** 2)
        if best2 is None or sse < best2[0]:
            best2 = (sse, j, t, lft.mean(), rgt.mean())
_, j2, t2, a2, b2 = best2
print(f"   лучший пень по остаткам: {names[j2]} ≤ {t2} → {a2:+.3f}, иначе {b2:+.3f}")
assert names[j2] == "этаж"
ex.note("""Второе дерево нашло то, что пропустило первое: квартиры на первом этаже дешевле примерно на 0.7 млн.
Именно так бустинг складывает простые деревья в сложную модель.""")

   остатки: -0.13  +0.77  -0.63  +0.20  +0.80  -1.00  +0.23  +0.83  -1.07  +0.00
   лучший пень по остаткам: этаж ≤ 1.5 → -0.708, иначе +0.472


> ▸ **Вывод.** Второе дерево нашло то, что пропустило первое: квартиры на первом этаже дешевле примерно на 0.7 млн. Именно так бустинг складывает простые деревья в сложную модель.

## Этап 5. Сверка с gbcourse.RegressionTree(max_depth=2)

In [7]:
lib = RegressionTree(max_depth=2).fit(X, price.mean() - price)   # дерево курса учится на градиентах g = F − y
lib_pred = price.mean() + lib.predict(X)                          # лист = −mean(g) = среднее остатков
print(lib.describe(feature_names=names))
print(f"   наибольшее расхождение прогнозов: {np.abs(lib_pred - pred).max():.1e}")
assert np.abs(lib_pred - pred).max() < 1e-12
ex.done()

├─ если площадь ≤ 64.000  (n = 10, выигрыш = 22.794)
│   ├─ если площадь ≤ 45.000  (n = 6, выигрыш = 3.521)
│   │   └─ лист: значение = -2.827 (n = 3)
│   ├─ иначе (площадь > 45.000)
│   │   └─ лист: значение = -0.660 (n = 3)
├─ иначе (площадь > 64.000)
│   ├─ если площадь ≤ 88.500  (n = 4, выигрыш = 4.950)
│   │   └─ лист: значение = 1.707 (n = 3)
│   ├─ иначе (площадь > 88.500)
│   │   └─ лист: значение = 5.340 (n = 1)
   наибольшее расхождение прогнозов: 1.8e-15
Готово за 0.2 с


## Попробуйте сами

Измените код в ячейках выше и выполните их заново:

1. Постройте дерево глубины 3 (`RegressionTree(max_depth=3)`) — появится ли этаж в разбиениях?
2. Для одного порога площади посчитайте SSE левой и правой частей вручную и сверьте с перебором.
3. Доведите второй шаг бустинга до конца: прибавьте пень по остаткам с темпом 0.5 и посчитайте новую MSE.

## Что дальше

- **Теория в курсе:** [2.1 «Пень»](../../../lessons/lesson_2_1/README.md), [2.2 «Критерии разбиения»](../../../lessons/lesson_2_2/README.md), [2.3 «Рост дерева»](../../../lessons/lesson_2_3/README.md).
- **Предыдущий пример:** [01. Градиентный бустинг вручную на 6 точках](01_boosting_by_hand.ipynb)
- **Следующий пример:** [03. Бустинг для классификации вручную: 8 объектов, логиты и шаг Ньютона](03_classification_by_hand.ipynb)
- **Все примеры:** [examples/README.md](../../README.md)